# GRF Training Notebook
**Google Research Football** -- `academy_3_vs_1_with_keeper`

Separate notebook because GRF's C++ engine must be compiled against the
running Python, and the compilation takes ~5 minutes on first run.

## Workflow
1. Run all cells top-to-bottom.
2. Results are saved to Google Drive automatically.
3. `mappo_demo.ipynb` loads them via `DEMO_MODE=True`.

In [ ]:
# ================================================================
# CONFIGURATION -- edit before running
# ================================================================
GITHUB_URL    = 'https://github.com/keckster00/mappo'
DRIVE_RESULTS = '/content/drive/MyDrive/mappo_demo'  # must match main notebook

GRF_STEPS = 5_000_000   # ~30-60 min on A100
N_THREADS = 64           # GRF envs are heavier than MPE; 64 is safe on A100

print('Configuration loaded.')
print(f'  GRF_STEPS : {GRF_STEPS:,}')
print(f'  N_THREADS : {N_THREADS}')


In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Drive mounted. Results folder: {DRIVE_RESULTS}')

REPO_PATH = '/content/mappo'
if not os.path.exists(REPO_PATH):
    print(f'Cloning repo from {GITHUB_URL}...')
    ret = os.system(f'git clone {GITHUB_URL} {REPO_PATH}')
    if ret != 0:
        raise RuntimeError('git clone failed. Check GITHUB_URL in the config cell.')
else:
    print(f'Repo already at {REPO_PATH}')

os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')

print('Pulling latest code...')
ret = os.system('git pull origin main')
if ret != 0:
    print('WARNING: git pull failed. Continuing with existing code.')


## Setup
Installs system libraries, GRF, PyTorch, and the onpolicy package.
**Run once per Colab session** (safe to re-run).

In [ ]:
import subprocess, sys, os

def run(cmd, label):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0
    print(f'  [{"OK" if ok else "FAIL"}] {label}')
    if not ok:
        if result.stdout.strip(): print('  stdout:', result.stdout[-3000:])
        if result.stderr.strip(): print('  stderr:', result.stderr[-3000:])
    return ok

def pip(*args):
    return run(f'{sys.executable} -m pip install -q ' + ' '.join(args), ' '.join(args[:2]))

# 1. System libraries required by GRF's C++ engine (from GRF README)
print('Installing system libraries...')
run('apt-get update -qq > /dev/null 2>&1', 'apt-get update')
run(
    'apt-get install -y '
    'git cmake build-essential '
    'libgl1-mesa-dev libglu1-mesa-dev '
    'libsdl2-dev libsdl2-image-dev libsdl2-ttf-dev libsdl2-gfx-dev '
    'libboost-all-dev '
    'xvfb x11vnc '
    '> /dev/null 2>&1',
    'system libraries'
)

# 2. Upgrade pip/setuptools/wheel as recommended by GRF README
print('Upgrading build tools...')
pip('--upgrade pip setuptools psutil wheel')

# 3. Install gfootball from PyPI (recommended install method per README)
print('Installing gfootball...')
ok = pip('gfootball')
if not ok:
    raise RuntimeError(
        'gfootball install failed. '
        'Check stderr above for details.'
    )

# 4. Pin gym to pre-0.26 so onpolicy gets the 4-tuple step() API it expects.
#    gfootball may have pulled in a newer gym as a dependency.
pip("'gym==0.25.2'")

# 5. ML training dependencies
print('Installing ML deps...')
pip('torch torchvision --index-url https://download.pytorch.org/whl/cu121')
pip('numpy==1.26.4 scipy imageio tensorboard tensorboardX setproctitle wandb')

# 6. Install onpolicy in editable mode
print('Installing onpolicy...')
pip(f'-e {REPO_PATH}')

print('All setup steps complete.')


In [ ]:
import sys, subprocess

print(f'Python: {sys.version}')
checks = [
    "import gfootball; print('gfootball ok')",
    "import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())",
    "import gym; print('gym', gym.__version__)",
    "import numpy as np; print('numpy', np.__version__)",
    "import onpolicy; print('onpolicy ok')",
]
all_ok = True
for check in checks:
    r = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'FAIL'
    out = (r.stdout or r.stderr).strip().splitlines()[0] if (r.stdout or r.stderr).strip() else ''
    print(f'  [{status}] {out}')
    if r.returncode != 0:
        all_ok = False
if all_ok:
    print('All checks passed. Safe to run training.')
else:
    raise RuntimeError('Checks failed -- fix errors above before training.')


## Training
A smoke test (1 thread, 2 000 steps) runs first to catch errors before the
full 30-60 min run. Expect ~30 s for the smoke test.

In [ ]:
import subprocess, sys, os, tempfile, shutil

os.environ['WANDB_MODE']     = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'
GRF_SCRIPT   = f'{REPO_PATH}/onpolicy/scripts/train/train_football.py'

# GRF requires a virtual display even in headless training
os.system('pkill Xvfb 2>/dev/null; Xvfb :99 -screen 0 1280x720x24 &')
env = os.environ.copy()
env['DISPLAY']    = ':99'
env['PYTHONPATH'] = REPO_PATH + ':' + env.get('PYTHONPATH', '')

COMMON_ARGS = [
    '--env_name',           'Football',
    '--scenario_name',      'academy_3_vs_1_with_keeper',
    '--num_agents',         '3',
    '--algorithm_name',     'rmappo',
    '--seed',               '1',
    '--n_training_threads', '1',
    '--num_mini_batch',     '1',
    '--episode_length',     '200',
    '--ppo_epoch',          '15',
    '--use_ReLU',
    '--gain',               '0.01',
    '--lr',                 '5e-4',
    '--critic_lr',          '5e-4',
    '--representation',     'simple115v2',
    '--rewards',            'scoring',
    '--use_wandb',          # store_false: disables wandb, uses TensorBoard
]

def run_grf(extra_args, label):
    cmd = [sys.executable, GRF_SCRIPT] + COMMON_ARGS + extra_args
    with tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False) as ef:
        err_path = ef.name
    with open(err_path, 'w') as ef:
        result = subprocess.run(cmd, stderr=ef, text=True, env=env)
    if result.returncode == 0:
        print(f'{label}: PASSED')
        os.unlink(err_path)
    else:
        print(f'{label}: FAILED (exit {result.returncode})')
        with open(err_path) as f:
            lines = f.readlines()
        print('--- last 60 lines of stderr ---')
        print(''.join(lines[-60:]))
        print('--- end ---')
        os.unlink(err_path)
    return result.returncode == 0

# Smoke test: 1 thread x 2 000 steps -- catches import/config errors fast
print('Running smoke test...')
ok = run_grf([
    '--experiment_name',    'smoke',
    '--n_rollout_threads',  '1',
    '--num_env_steps',      '2000',
    '--ppo_epoch',          '1',
    '--save_interval',      '9999',
    '--log_interval',       '1',
], 'Smoke test')
if not ok:
    raise RuntimeError('Smoke test failed -- fix the error above before full training.')
print('Smoke test passed. Starting full training...')
print(f'  steps={GRF_STEPS:,}  threads={N_THREADS}')
print('-' * 60)

ok = run_grf([
    '--experiment_name',    'demo',
    '--n_rollout_threads',  str(N_THREADS),
    '--num_env_steps',      str(GRF_STEPS),
    '--save_interval',      '25',
    '--log_interval',       '25',
], 'Full training')

print('-' * 60)
if ok:
    print('Training complete!')
else:
    print('Training FAILED -- check stderr above.')


In [ ]:
import shutil, os

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'
src = f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo'
dst = f'{DRIVE_RESULTS}/GRF_3v1'

if os.path.exists(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Results saved to: {dst}')
    print('Load in mappo_demo.ipynb: set DEMO_MODE=True and run the GRF cell.')
else:
    print(f'No results found at {src}')
    print('Check that training completed successfully.')


## Results -- Training Curve

In [ ]:
import os, glob
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'

def load_scalar(log_dir, tag):
    nested = os.path.join(log_dir, tag, tag)
    target = nested if os.path.isdir(nested) else log_dir
    ea = EventAccumulator(target)
    ea.Reload()
    if tag in ea.Tags().get('scalars', []):
        ev = ea.Scalars(tag)
        return [e.step for e in ev], [e.value for e in ev]
    return [], []

log_dirs = sorted(glob.glob(
    f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo/run*/logs'))

if not log_dirs:
    print('No logs found. Run the training cell first.')
else:
    steps, values = load_scalar(log_dirs[-1], 'average_episode_rewards')
    if steps:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(steps, values, color='#4CAF50', linewidth=2)
        ax.set_xlabel('Environment Steps', fontsize=12)
        ax.set_ylabel('Average Episode Reward', fontsize=12)
        ax.set_title('MAPPO -- GRF academy_3_vs_1_with_keeper', fontsize=14, fontweight='bold')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        fig_path = f'{DRIVE_RESULTS}/tc2_grf_curve.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f'Final reward: {values[-1]:.4f}')
        print(f'Figure saved: {fig_path}')
        plt.show()
    else:
        print('No scalar data found in', log_dirs[-1])
        ea_root = EventAccumulator(log_dirs[-1]); ea_root.Reload()
        print('Available tags:', ea_root.Tags().get('scalars', []))
        for sub in sorted(os.listdir(log_dirs[-1])):
            print(' ', sub)
